# vader

> Extract sentiment scores from a text using VADER.

Textplumber implements feature extraction using [VADER](https://github.com/cjhutto/vaderSentiment), "a lexicon and rule-based sentiment analysis tool". [VADER's GitHub repository](https://github.com/cjhutto/vaderSentiment) and [Hutto and Gilbert's 2014 paper about VADER](https://ojs.aaai.org/index.php/ICWSM/article/view/14550) are the best explanation of VADER and how the VADER lexicon and rules were derived.   

This functionality is not available in the latest version available on Pypi (0.0.8), but will be released as part of version 0.0.9.

In [ ]:
#| default_exp vader

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from textplumber.store import TextFeatureStore
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import numpy as np
from fastcore.basics import patch
from nltk.tokenize import sent_tokenize
import nltk

In [ ]:
#| export
class VaderSentimentExtractor(BaseEstimator, TransformerMixin):
	""" Sci-kit Learn pipeline component to extract sentiment features using VADER. """
	def __init__(self, 
			  feature_store:TextFeatureStore = None, # (not implemented currently)
			  output:str = 'polarity', # 'polarity' (VADER's compound score), 'proportions' (ratios for proportions of text that are positive, neutral or negative), or 'allstats' (equivalent to 'polarity' + 'proportions'), 'labels' (positive, neutral, negative), profile (for a document sentiment profile vector consisting of document-level and sentence-level features with their order in the document represented)
			  neutral_threshold:float = 0.05, # threshold for neutral sentiment
			  profile_first_n:int = 3, # number of sentences at start of doc to profile
			  profile_last_n:int = 3, # number of sentences at end of doc to profile
			  profile_sample_n:int = 4, # number of sentences to sample from doc sentences after first and last removed
			  profile_min_sentence_chars:int = 10, # minimum number of characters in body sentences to be included in the profile
			):
		
		self.feature_store = feature_store
		if output not in ['polarity', 'proportions', 'allstats', 'labels', 'profile']:
			raise ValueError(f"output must be one of ['polarity', 'proportions', 'allstats', 'labels', 'profile'], got {output}")
		self.output = output
		self.neutral_threshold = neutral_threshold
		self.profile_first_n = profile_first_n
		self.profile_last_n = profile_last_n
		self.profile_sample_n = profile_sample_n
		self.profile_min_sentence_chars = profile_min_sentence_chars
		if self.output == 'profile':
			# seeding random number generator for reproducibility
			np.random.seed(55)
			try:
				nltk.data.find('tokenizers/punkt_tab')
			except LookupError:
				nltk.download('punkt_tab')

By default `neutral_threshold` is set to 0.05. This means that any text with polarity greater than -0.05 and less than 0.05 will be 'neutral'. The 0.05 value default is the recommendation of the [VADER Github page](https://github.com/cjhutto/vaderSentiment?tab=readme-ov-file#about-the-scoring), but this can be tuned as needed. 

In [ ]:
#| export
@patch
def fit(self:VaderSentimentExtractor, X, y=None):
	""" Fit is implemented, but does nothing. """
	return self

In [ ]:
#| export
@patch
def convert_score_to_label(self:VaderSentimentExtractor, score: float, label_mapping = None) -> str:
	""" Convert VADER score to label. """
	if score >= self.neutral_threshold:
		label = 'positive'
	elif score <= self.neutral_threshold * -1:
		label = 'negative'
	else:
		label = 'neutral'
	if label_mapping is not None:
		label = label_mapping[label]
	return label

In [ ]:
#| export
@patch
def convert_scores_to_labels(self:VaderSentimentExtractor, scores: list[float], label_mapping = None):
	""" Convert VADER score to label. """
	for score in scores:
		yield self.convert_score_to_label(score)

In [ ]:
#| export
@patch
def profile(self:VaderSentimentExtractor, 
				text: str, # the document text
				doc_level_scores: dict, # VADER scores for document text
				) -> list[float]: # a document profile vector consisting of the document level scores and sentence-level scores across the document
	""" Create a document profile with VADER scores, which makes use of document level scores and sentence-level scores across the document. """
	sentences = sent_tokenize(text)
	scores = [doc_level_scores['compound'], doc_level_scores['neg'], doc_level_scores['neu'], doc_level_scores['pos']]
	sentences_to_score = []
	if len(sentences) < self.profile_first_n + self.profile_last_n + self.profile_sample_n:
		if len(sentences) < self.profile_first_n: ## padding to end of start if needed
			sentences = sentences + [''] * (self.profile_first_n + self.profile_last_n + self.profile_sample_n - len(sentences))
		elif len(sentences) < self.profile_first_n + self.profile_last_n: # # padding to start of end if needed
			sentences = sentences[:self.profile_first_n] + [''] * (self.profile_first_n + self.profile_last_n + self.profile_sample_n - len(sentences)) + sentences[self.profile_first_n:]
		else:
			sentences = sentences[:self.profile_first_n] + sentences[self.profile_first_n:-self.profile_last_n] + [''] * (self.profile_first_n + self.profile_last_n + self.profile_sample_n - len(sentences)) + sentences[-self.profile_last_n:]

	if self.profile_min_sentence_chars > 0 and len(sentences) > self.profile_first_n + self.profile_last_n + self.profile_sample_n:
		overlap = len(sentences) - self.profile_first_n - self.profile_last_n - self.profile_sample_n
		for i in range(self.profile_first_n, len(sentences) - self.profile_last_n):
			if len(sentences[i].strip()) < self.profile_min_sentence_chars:
				sentences[i] = None
				overlap -= 1
				if overlap == 0:
					break
		sentences = [sentence for sentence in sentences if sentence is not None]

	sentences_to_score.extend(sentences[:self.profile_first_n])
	sentences_to_score.extend(sentences[-self.profile_last_n:])
	sentences = sentences[self.profile_first_n:-self.profile_last_n]
	if len(sentences) == self.profile_sample_n:
		sentences_to_score.extend(sentences)
	elif len(sentences) > self.profile_sample_n:
		sample_indices = np.random.choice(len(sentences), self.profile_sample_n, replace=False)
		sample_indices.sort()
		sentences_to_score.extend([sentences[i] for i in sample_indices])
	del sentences

	for i, sentence in enumerate(sentences_to_score):
		scores.append(self.analyzer_.polarity_scores(sentence)['compound'])

	if len(scores) < 4 + self.profile_first_n + self.profile_last_n + self.profile_sample_n:
		print('scores')
		print(scores)
		print('sentences to score')	
		print(sentences_to_score)
		print('full text')
		print(text)
		raise ValueError(f"VADER profile vector is too short")

	return scores

In [ ]:
#| export
@patch
def transform(self:VaderSentimentExtractor, X):
	""" Extracts the sentiment from the text using VADER. """
	results = []
	self.analyzer_ = SentimentIntensityAnalyzer()
	for text in X:
		scores = self.analyzer_.polarity_scores(text)
		if self.output == 'proportions':
			results.append([scores['pos'], scores['neu'], scores['neg']])
		elif self.output == 'labels':
			compound = scores['compound']
			results.append(self.convert_score_to_label(compound))
		elif self.output == 'allstats':
			results.append([scores['pos'], scores['neu'], scores['neg'], scores['compound']])
		elif self.output == 'profile':
			results.append(self.profile(text, scores))
		else: # default
			results.append([scores['compound']])
	return np.atleast_2d(results)  # Ensure the output is always a 2D array

In [ ]:
#| export
@patch
def get_feature_names_out(self:VaderSentimentExtractor, input_features=None):
	""" Get the feature names out from the model. """
	if self.output == 'proportions':
		return ['positive', 'neutral', 'negative']
	elif self.output == 'labels':
		return ['label']
	elif self.output == 'allstats':
		return ['positive', 'neutral', 'negative', 'compound']
	elif self.output == 'profile':
		return ['doc_compound', 'doc_negative', 'doc_neutral', 'doc_positive'] + [f'introduction_sentence_{i}' for i in range(self.profile_first_n)] + [f'conclusion_sentence_{i}' for i in range(self.profile_last_n)] + [f'body_sentence_sample_{i}' for i in range(self.profile_sample_n)]
	else: # default
		return ['polarity']


In [ ]:
#| hide
# from imdb dataset hf
# test_text = """
# I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, even then it's not shot like some cheaply made porno. While my countrymen mind find it shocking, in reality sex and nudity are a major staple in Swedish cinema. Even Ingmar Bergman, arguably their answer to good old boy John Ford, had sex scenes in his films.<br /><br />I do commend the filmmakers for the fact that any sex shown in the film is shown for artistic purposes rather than just to shock people and make money to be shown in pornographic theaters in America. I AM CURIOUS-YELLOW is a good film for anyone wanting to study the meat and potatoes (no pun intended) of Swedish cinema. But really, this film doesn't have much of a plot.
# """
# test_text = """
# This movie was just plain bad. Just about every cop movie cliché is present and accounted for. Bad guy gets away? check. Partner? check. Wacky personality clash with partner? check. Rookie with something to prove? check. Rookie shows up grizzled veteran. check. About the only ones it didn't touch on were idiot shoot themselves in the foot and retirony but I guess they're saving those old chestnuts for Dooley's next outing. Add in the battle of the sexes with Girl Power along with tired old sight gags and banal overdone material like Dooley's prize car getting trashed all the time and you have the recipe for one really bad movie. Avoid this one at all costs.
# """
# extractor = VaderSentimentExtractor(output='profile')
# print(extractor.transform([test_text]))
# extractor = VaderSentimentExtractor(output='profile', profile_min_sentence_chars=1)
# test_texts = ['good. good.', 'good. good. good. bad. ', 'good. good. good. bad. bad. bad.', 'good. good. good. great. bad. bad. bad.', 'good. good. good. great. great. great. bad. bad. bad.', 'good. good. good. great. greater. greatest. very great. very greatest. bad. bad. bad.']
# print(extractor.transform(test_texts))

In [ ]:
#| hide
vader_extractor = VaderSentimentExtractor(output = 'labels')
vader_extractor.fit(['Hello, world! Today is great!'])
assert vader_extractor.get_feature_names_out() == ['label']
assert vader_extractor.transform(['Hello, world! Today is great!']) == ['positive']
vader_extractor = VaderSentimentExtractor(output = 'proportions')
vader_extractor.fit(['Hello, world! Today is great!'])
assert vader_extractor.get_feature_names_out() == ['positive', 'neutral', 'negative']
test_texts = ['good, good, good, good', 'bad, bad, bad, bad', 'fish, cat, box, dog']
assert tuple(vader_extractor.transform(test_texts)[0]) == tuple([1.0, 0.0, 0.0])
assert tuple(vader_extractor.transform(test_texts)[1]) == tuple([0.0, 0.0, 1.0])
assert tuple(vader_extractor.transform(test_texts)[2]) == tuple([0.0, 1.0, 0.0])
vader_extractor = VaderSentimentExtractor(output = 'polarity')
assert vader_extractor.get_feature_names_out() == ['polarity']
assert vader_extractor.transform(test_texts)[0] > 0.05
assert vader_extractor.transform(test_texts)[1] < -0.05
assert vader_extractor.transform(test_texts)[2] == 0

del vader_extractor


In [ ]:
#| export
class VaderSentimentEstimator(VaderSentimentExtractor, ClassifierMixin):
	""" Sci-kit Learn pipeline component to predict sentiment using VADER. """

	def __init__(self,
				 output:str = 'labels', # 'polarity' (VADER's compound score) or 'labels' (positive, neutral, negative)
				 neutral_threshold:float = 0.05, # threshold for neutral sentiment (see note for VaderSentimentExtractor)
				 label_mapping:dict|None = None, # (ignored if labels is None) mapping of labels to desired labels - keys should be 'positive', 'neutral', 'negative' and values should be desired labels
				 ):
		
		super().__init__()
		if output not in ['polarity', 'labels']:
			raise ValueError(f"output must be one of ['polarity', 'labels'], got {output}")
		self.output = output
		self.label_mapping = label_mapping
		self.neutral_threshold = neutral_threshold

If `output` is set to `labels` then `VaderSentimentEstimator` functions as a pseudo-classifier. With `output` set to `polarity`, it functions as a regressor or scorer.

By default the VaderSentimentEstimator is setup to work with three classes (i.e. it returns 'positive', 'neutral' or 'negative'). If you only have two classes (negative/positive), set the neutral_threshold to 0 to remove the neutral class. Even slightly negative or positive scores will be assigned a non-neutral label. A 0 polarity score will be assigned to positive in the two-class case. The classes might be better interpreted as negative and not-negative in this instance.

	neutral_threshold = 0

You may also want/need to create a label mapping so that the correct class IDs are returned by the estimator.  

For example ...

	label_mapping = {
		'positive': 1,
		'negative': 0
	}



In [ ]:
#| export
@patch
def predict(self:VaderSentimentEstimator, X):
	""" Predict the sentiment of texts using VADER. """
	y_predicted = self.transform(X).ravel()
	if self.output == 'labels' and self.label_mapping is not None:
		for i, prediction in enumerate(y_predicted):
			y_predicted[i] = self.label_mapping[prediction]
		dtype = type(list(self.label_mapping.values())[0])
	elif self.output == 'labels':
		dtype = str
	else:
		dtype = float
	return np.array(y_predicted, dtype=dtype)

In [ ]:
#| hide
vader_estimator = VaderSentimentEstimator(output = 'labels')
test_texts = ['good, good, good, good', 'bad, bad, bad, bad', 'fish, cat, box, dog']
vader_estimator.fit(test_texts)
assert tuple(vader_estimator.predict(test_texts)) == tuple(['positive', 'negative', 'neutral'])

vader_estimator = VaderSentimentEstimator(output = 'labels')
vader_estimator.fit(test_texts)
assert tuple(vader_estimator.predict(test_texts)) == tuple(['positive', 'negative', 'neutral'])

vader_estimator = VaderSentimentEstimator(output = 'labels', label_mapping = {'positive': 2, 'neutral': 1, 'negative': 0})
vader_estimator.fit(test_texts)
assert tuple(vader_estimator.predict(test_texts)) == tuple([2, 0, 1])

# two class example # i.e. negative or not-negative
vader_estimator = VaderSentimentEstimator(output = 'labels', neutral_threshold=0, label_mapping = {'positive': 1, 'negative': 0})
vader_estimator.fit(test_texts)
assert tuple(vader_estimator.predict(test_texts)) == tuple([1, 0, 1]) 

vader_estimator = VaderSentimentEstimator(output = 'polarity')
vader_estimator.fit(test_texts)
assert vader_estimator.predict(test_texts)[0] > 0.05
assert vader_estimator.predict(test_texts)[1] < -0.05
assert vader_estimator.predict(test_texts)[2] == 0

del vader_estimator


## Example using VADER as an estimator

Here is an example demonstrating how to use `VaderSentimentEstimator` in a pipeline. 

In [ ]:
#| hide
import os

In [ ]:
#| hide
if os.path.exists('feature_store_example_vader.sqlite'):
	os.remove('feature_store_example_vader.sqlite')

In [ ]:
#| hide
if os.path.exists('results_example_vader.csv'):
	os.remove('results_example_vader.csv')

In [ ]:
#| eval: false
from textplumber.report import plot_confusion_matrix, get_label_names, save_results, plot_logistic_regression_features_from_pipeline
from textplumber.vader import VaderSentimentEstimator, VaderSentimentExtractor
from textplumber.preprocess import SpacyPreprocessor
from textplumber.embeddings import Model2VecEmbedder
from textplumber.core import get_stop_words

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from datasets import load_dataset
from sklearn.model_selection import train_test_split

import pandas as pd

Here we load text samples from a [sentiment dataset](https://huggingface.co/datasets/cardiffnlp/tweet_eval). The dataset has validation and test sets, but we are just working with the train split in this instance.

In [ ]:
#| eval: false
dataset_name = 'cardiffnlp/tweet_eval'
dataset_dir = 'sentiment'
dataset = load_dataset(dataset_name, data_dir = dataset_dir, split='train')

Only getting 5000 for each class ...

In [ ]:
#| eval: false
label_column = 'label'
target_names = get_label_names(dataset, label_column)
target_classes = list(range(len(target_names)))

# selecting 5000 per class here ...
df = dataset.to_pandas()
sampled_dfs = [
    group.sample(n=5000, random_state=42)
    for _, group in df.groupby('label')
]
df = pd.concat(sampled_dfs, ignore_index=True)

X = df['text']
y = df[label_column]

In [ ]:
#| eval: false
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
#| eval: false
pd.set_option('display.max_colwidth', 200)
df = pd.DataFrame({'text': X_train, 'label': y_train})
df['label_name'] = df['label'].apply(lambda x: target_names[x])
df.head(5)

,text,label,label_name
5497,Harper walks. #Nats now have two men on with one out. Escobar will bat next. Still 8-7 Mets in the bottom of the 9th.,1,neutral
12677,May or may not be getting ready to watch the very first season EVER of Big Brother. @user I'm so ready to see what Julie looks like!,2,positive
8037,Patrick Leahy & Christian Bale together again. U.S. Senator to make his 2nd cameo in Batman movie in Dark Knight Rises,1,neutral
6670,"""A smartphone review that the tech press needs to read twice, obviously great for consumers too: Moto G 3rd gen review",1,neutral
79,"Armed with the tools of power &amp; intimidation Harper is trying to steal our country. On October 19th, armed with pencils LET'S TAKE IT BACK.",0,negative


Setup a label mapping so that the desired label values are returned rather than the default 'positive', 'neutral', 'negative' labels.

In [ ]:
#| eval: false
label_mapping = {
	'negative': 0,
	'neutral': 1, 
	'positive': 2
}

Our pipeline only has one component! Notice that the label_mapping is passed to the estimator to ensure we return comparable labels to the data-set labels.

In [ ]:
#| eval: false
pipeline = Pipeline([
    ('vader_estimator', VaderSentimentEstimator(output = 'labels', label_mapping = label_mapping)),
], verbose=True)

display(pipeline)

Pipeline(steps=[('vader_estimator',
                 VaderSentimentEstimator(label_mapping={'negative': 0,
                                                        'neutral': 1,
                                                        'positive': 2}))],
         verbose=True)

Note: fit is not really required, because VADER is based on heuristics that are independent of the training data.

In [ ]:
#| eval: false
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

[Pipeline] ... (step 1 of 1) Processing vader_estimator, total=   0.0s


Log some results for the final cell summary.

In [ ]:
#| eval: false
dataset_descriptor = 'tweet_eval sentiment dataset, 5000 rows per class randomly sampled from train split (per class - 4000 train, 1000 test)'
experiment_descriptor = 'Assigning labels using VADER'
results = save_results('results_example_vader.csv', pipeline, experiment_descriptor, dataset_descriptor, y_test, y_pred, target_classes, target_names, classifier_step_name = 'vader_estimator')

Here are the results ...

In [ ]:
#| eval: false
print(classification_report(y_test, y_pred, labels = target_classes, target_names = target_names, digits=3))
plot_confusion_matrix(y_test, y_pred, target_classes, target_names)

              precision    recall  f1-score   support

    negative      0.657     0.591     0.622      1000
     neutral      0.533     0.392     0.452      1000
    positive      0.514     0.702     0.594      1000

    accuracy                          0.562      3000
   macro avg      0.568     0.562     0.556      3000
weighted avg      0.568     0.562     0.556      3000



## Example using VADER as a feature extractor

Here is an example demonstrating how to use `VaderSentimentExtractor` in a pipeline. We will use the same dataset as above so we can compare results.

The VaderSentimentExtractor can return VADER's compound polarity score ('polarity'), proportions of positive/neutral/negative ('proportions'), or all statistics ('allstats').  

Note: when proportions are being returned by VADER, these do not make use of rules reflected in the compound polarity score. See [VADER's Github documentation](https://github.com/cjhutto/vaderSentiment?tab=readme-ov-file#about-the-scoring) for more information. 

In [ ]:
#| eval: false
pipeline = Pipeline([
    ('vader_extractor', VaderSentimentExtractor(output = 'allstats')),
    ('classifier', LogisticRegression(max_iter = 5000, random_state=55))
], verbose=True)

display(pipeline)

Pipeline(steps=[('vader_extractor', VaderSentimentExtractor(output='allstats')),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=55))],
         verbose=True)

In [ ]:
#| eval: false
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

[Pipeline] ... (step 1 of 2) Processing vader_extractor, total=   0.4s
[Pipeline] ........ (step 2 of 2) Processing classifier, total=   0.0s


Logging results for summary ...

In [ ]:
#| eval: false
experiment_descriptor = 'Logistic Regression classifier using VADER features'
results = save_results('results_example_vader.csv', pipeline, experiment_descriptor, dataset_descriptor, y_test, y_pred, target_classes, target_names, classifier_step_name = 'classifier')

As might be expected, the performance of a classifier based on statistical output from VADER is similar to the results of the first experiment, which used VADER to assign the class ... 

In [ ]:
#| eval: false
print(classification_report(y_test, y_pred, labels = target_classes, target_names = target_names, digits=3))
plot_confusion_matrix(y_test, y_pred, target_classes, target_names)

              precision    recall  f1-score   support

    negative      0.660     0.632     0.646      1000
     neutral      0.527     0.493     0.510      1000
    positive      0.570     0.631     0.599      1000

    accuracy                          0.585      3000
   macro avg      0.586     0.585     0.585      3000
weighted avg      0.586     0.585     0.585      3000



It is anticipated that the VaderSentimentExtractor component will be used alongside other features. Here is an example where the VADER statistics are supplemented by other features.

Setup a feature store to save preprocessed texts ...

In [ ]:
#| eval: false
feature_store = TextFeatureStore('feature_store_example_vader.sqlite')

Augment the VADER features with embeddings ...

In [ ]:
#| eval: false
pipeline = Pipeline([
		('preprocess', SpacyPreprocessor(feature_store=feature_store)),
		('features', FeatureUnion([
				('vader', VaderSentimentExtractor(output = 'allstats')),
				('embeddings', Model2VecEmbedder(feature_store=feature_store)),
		], verbose=True)),
		('classifier', LogisticRegression(max_iter = 5000, random_state=55)),
], verbose=True)

display(pipeline)

Pipeline(steps=[('preprocess',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object>)),
                ('features',
                 FeatureUnion(transformer_list=[('vader',
                                                 VaderSentimentExtractor(output='allstats')),
                                                ('embeddings',
                                                 Model2VecEmbedder(feature_store=<textplumber.store.TextFeatureStore object>))],
                              verbose=True)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=55))],
         verbose=True)

In [ ]:
#| eval: false
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

[Pipeline] ........ (step 1 of 3) Processing preprocess, total=  11.0s
[FeatureUnion] ......... (step 1 of 2) Processing vader, total=   0.4s
[FeatureUnion] .... (step 2 of 2) Processing embeddings, total=   0.5s
[Pipeline] .......... (step 2 of 3) Processing features, total=   0.8s
[Pipeline] ........ (step 3 of 3) Processing classifier, total=   8.1s


Logging results for summary ...

In [ ]:
#| eval: false
experiment_descriptor = 'Logistic Regression classifier using VADER features and Model2Vec embeddings'
results = save_results('results_example_vader.csv', pipeline, experiment_descriptor, dataset_descriptor, y_test, y_pred, target_classes, target_names, classifier_step_name = 'classifier')

In [ ]:
#| eval: false
print(classification_report(y_test, y_pred, labels = target_classes, target_names = target_names, digits=3))
plot_confusion_matrix(y_test, y_pred, target_classes, target_names)

              precision    recall  f1-score   support

    negative      0.704     0.725     0.714      1000
     neutral      0.594     0.574     0.584      1000
    positive      0.663     0.666     0.665      1000

    accuracy                          0.655      3000
   macro avg      0.654     0.655     0.654      3000
weighted avg      0.654     0.655     0.654      3000



Do the VADER statistics contribute to the accuracy? In other words, could we remove the VADER stats for similar performance? This sanity check is run in the next cell. Results are shown in the results summary.

In [ ]:
#| eval: false
pipeline = Pipeline([
		('preprocess', SpacyPreprocessor(feature_store=feature_store)),
		('features', FeatureUnion([
				('embeddings', Model2VecEmbedder(feature_store=feature_store)),
		], verbose=True)),
		('classifier', LogisticRegression(max_iter = 5000, random_state=55)),
], verbose=True)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

experiment_descriptor = 'Logistic Regression classifier using only Model2Vec embeddings (Sanity Check)'
results = save_results('results_example_vader.csv', pipeline, experiment_descriptor, dataset_descriptor, y_test, y_pred, target_classes, target_names, classifier_step_name = 'classifier')


[Pipeline] ........ (step 1 of 3) Processing preprocess, total=   0.4s
[FeatureUnion] .... (step 1 of 1) Processing embeddings, total=   0.5s
[Pipeline] .......... (step 2 of 3) Processing features, total=   0.5s
[Pipeline] ........ (step 3 of 3) Processing classifier, total=   5.5s


Results for the four experiments are shown below. The best model achieves an F1 score of 0.655. While much better performance is possible for sentiment classification and only a small amount of training data was used, the model with VADER statistics *and* embeddings outperformed:  

* a model trained on embeddings alone  
* a model trained on VADER's statistical output  
* the predictions of the VADER algorithm itself.


In [ ]:
#| eval: false
fields = ['experiment', 'accuracy_f1', 'negative_f1', 'neutral_f1', 'positive_f1']
display(pd.read_csv('results_example_vader.csv').sort_values(by='accuracy_f1', ascending=False)[fields])

,experiment,accuracy_f1,negative_f1,neutral_f1,positive_f1
2,Logistic Regression classifier using VADER features and Model2Vec embeddings,0.655000,0.714286,0.583927,0.664671
3,Logistic Regression classifier using only Model2Vec embeddings (Sanity Check),0.634333,0.680997,0.559836,0.659023
1,Logistic Regression classifier using VADER features,0.585333,0.645557,0.509561,0.598956
0,Assigning labels using VADER,0.561667,0.622105,0.451873,0.593658


In [ ]:
#| hide
if os.path.exists('feature_store_example_vader.sqlite'):
	os.remove('feature_store_example_vader.sqlite')

In [ ]:
#| hide
if os.path.exists('results_example_vader.csv'):
	os.remove('results_example_vader.csv')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()